In [6]:
import numpy as np
import pandas as pd
import re
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score # used to find accuracy of model
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.stem.porter import PorterStemmer


In [2]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [7]:
sonar_df = pd.read_csv('/content/drive/MyDrive/sonar data.csv' , header = None)
sonar_df.head()

,0,1,2,3,4,5,6,7,8,9,...,51,52,53,54,55,56,57,58,59,60
0,0.0200,0.0371,0.0428,0.0207,0.0954,0.0986,0.1539,0.1601,0.3109,0.2111,...,0.0027,0.0065,0.0159,0.0072,0.0167,0.0180,0.0084,0.0090,0.0032,R
1,0.0453,0.0523,0.0843,0.0689,0.1183,0.2583,0.2156,0.3481,0.3337,0.2872,...,0.0084,0.0089,0.0048,0.0094,0.0191,0.0140,0.0049,0.0052,0.0044,R
2,0.0262,0.0582,0.1099,0.1083,0.0974,0.2280,0.2431,0.3771,0.5598,0.6194,...,0.0232,0.0166,0.0095,0.0180,0.0244,0.0316,0.0164,0.0095,0.0078,R
3,0.0100,0.0171,0.0623,0.0205,0.0205,0.0368,0.1098,0.1276,0.0598,0.1264,...,0.0121,0.0036,0.0150,0.0085,0.0073,0.0050,0.0044,0.0040,0.0117,R
4,0.0762,0.0666,0.0481,0.0394,0.0590,0.0649,0.1209,0.2467,0.3564,0.4459,...,0.0031,0.0054,0.0105,0.0110,0.0015,0.0072,0.0048,0.0107,0.0094,R


In [8]:
sonar_df.shape

(208, 61)

In [10]:
sonar_df.isnull().sum()

,0
0,0
1,0
2,0
3,0
4,0
...,...
56,0
57,0
58,0
59,0


In [13]:
sonar_df[60].value_counts()

,count
60,
M,111
R,97


In [15]:
# Grouping with respect to M/R & taking out mean
sonar_df.groupby(60).mean()

,0,1,2,3,4,5,6,7,8,9,...,50,51,52,53,54,55,56,57,58,59
60,,,,,,,,,,,,,,,,,,,,,
M,0.034989,0.045544,0.050720,0.064768,0.086715,0.111864,0.128359,0.149832,0.213492,0.251022,...,0.019352,0.016014,0.011643,0.012185,0.009923,0.008914,0.007825,0.009060,0.008695,0.006930
R,0.022498,0.030303,0.035951,0.041447,0.062028,0.096224,0.114180,0.117596,0.137392,0.159325,...,0.012311,0.010453,0.009640,0.009518,0.008567,0.007430,0.007814,0.006677,0.007078,0.006024


In [22]:
# Seperating data & labels into x & y :
x = sonar_df.drop(columns = 60 , axis = 1 )
y = sonar_df[60]

In [17]:
x

,0,1,2,3,4,5,6,7,8,9,...,50,51,52,53,54,55,56,57,58,59
0,0.0200,0.0371,0.0428,0.0207,0.0954,0.0986,0.1539,0.1601,0.3109,0.2111,...,0.0232,0.0027,0.0065,0.0159,0.0072,0.0167,0.0180,0.0084,0.0090,0.0032
1,0.0453,0.0523,0.0843,0.0689,0.1183,0.2583,0.2156,0.3481,0.3337,0.2872,...,0.0125,0.0084,0.0089,0.0048,0.0094,0.0191,0.0140,0.0049,0.0052,0.0044
2,0.0262,0.0582,0.1099,0.1083,0.0974,0.2280,0.2431,0.3771,0.5598,0.6194,...,0.0033,0.0232,0.0166,0.0095,0.0180,0.0244,0.0316,0.0164,0.0095,0.0078
3,0.0100,0.0171,0.0623,0.0205,0.0205,0.0368,0.1098,0.1276,0.0598,0.1264,...,0.0241,0.0121,0.0036,0.0150,0.0085,0.0073,0.0050,0.0044,0.0040,0.0117
4,0.0762,0.0666,0.0481,0.0394,0.0590,0.0649,0.1209,0.2467,0.3564,0.4459,...,0.0156,0.0031,0.0054,0.0105,0.0110,0.0015,0.0072,0.0048,0.0107,0.0094
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
203,0.0187,0.0346,0.0168,0.0177,0.0393,0.1630,0.2028,0.1694,0.2328,0.2684,...,0.0203,0.0116,0.0098,0.0199,0.0033,0.0101,0.0065,0.0115,0.0193,0.0157
204,0.0323,0.0101,0.0298,0.0564,0.0760,0.0958,0.0990,0.1018,0.1030,0.2154,...,0.0051,0.0061,0.0093,0.0135,0.0063,0.0063,0.0034,0.0032,0.0062,0.0067
205,0.0522,0.0437,0.0180,0.0292,0.0351,0.1171,0.1257,0.1178,0.1258,0.2529,...,0.0155,0.0160,0.0029,0.0051,0.0062,0.0089,0.0140,0.0138,0.0077,0.0031
206,0.0303,0.0353,0.0490,0.0608,0.0167,0.1354,0.1465,0.1123,0.1945,0.2354,...,0.0042,0.0086,0.0046,0.0126,0.0036,0.0035,0.0034,0.0079,0.0036,0.0048


In [18]:
y

,60
0,R
1,R
2,R
3,R
4,R
...,...
203,M
204,M
205,M
206,M


In [23]:
# train test spliting
x_train , x_test , y_train , y_test = train_test_split( x , y , test_size = 0.1  , random_state =2  , stratify=y )


In [24]:
# shape of all
print(x.shape , x_train.shape , x_test.shape)


(208, 60) (187, 60) (21, 60)


In [26]:
model = LogisticRegression()

In [28]:
# model training

model.fit(x_train, y_train)


LogisticRegression()

In [40]:
x_train_prediction = model.predict(x_train)
training_data_accuracy = accuracy_score(x_train_prediction , y_train)

In [41]:
print("Acc on training data : " , training_data_accuracy*100,"%")

Acc on training data :  81.81818181818183 %


In [42]:
x_test_prediction = model.predict(x_test)
test_data_accuracy = accuracy_score(x_test_prediction, y_test)

In [43]:
print("Acc on test data : " , test_data_accuracy*100,"%")

Acc on test data :  90.47619047619048 %


In [44]:
input_data = (0.034989, 0.045544, 0.050720, 0.064768, 0.086715, 0.111864, 0.128359, 0.149832, 0.213492, 0.251022, 0.289581, 0.301459, 0.314426, 0.320692, 0.331182, 0.380999, 0.415007, 0.455882, 0.538062, 0.617941, 0.667426, 0.672325, 0.676701, 0.689165, 0.681204, 0.706075, 0.714754, 0.712269, 0.650283, 0.581796, 0.482378, 0.428049, 0.396577, 0.36614, 0.337553, 0.318553, 0.317034, 0.331608, 0.336365, 0.305221, 0.292594, 0.300975, 0.276883, 0.248106, 0.245225, 0.198804, 0.146917, 0.110594, 0.063708, 0.022721, 0.019352, 0.016014, 0.011643, 0.012185, 0.009923, 0.008914, 0.007825, 0.009060, 0.008695, 0.006930)

In [45]:
# convert input_data into numpy array
input_arr = np.array(input_data)

In [46]:
input_arr

array([0.034989, 0.045544, 0.05072 , 0.064768, 0.086715, 0.111864,
       0.128359, 0.149832, 0.213492, 0.251022, 0.289581, 0.301459,
       0.314426, 0.320692, 0.331182, 0.380999, 0.415007, 0.455882,
       0.538062, 0.617941, 0.667426, 0.672325, 0.676701, 0.689165,
       0.681204, 0.706075, 0.714754, 0.712269, 0.650283, 0.581796,
       0.482378, 0.428049, 0.396577, 0.36614 , 0.337553, 0.318553,
       0.317034, 0.331608, 0.336365, 0.305221, 0.292594, 0.300975,
       0.276883, 0.248106, 0.245225, 0.198804, 0.146917, 0.110594,
       0.063708, 0.022721, 0.019352, 0.016014, 0.011643, 0.012185,
       0.009923, 0.008914, 0.007825, 0.00906 , 0.008695, 0.00693 ])

In [48]:
# reshape the np array as we are predicting for one instance
input_reshaped = input_arr.reshape(1,-1)

In [53]:
prediction = model.predict(input_reshaped)
if prediction == "M":
  print("Mine")
else:
  print("Rock")

Mine
